# Data Cleaning Pipeline - E-Commerce Sales Analytics

This notebook documents the data cleaning and transformation process:
- Schema validation
- Missing value handling
- Duplicate removal
- Date standardization
- Outlier detection
- Data type conversion

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Load Raw Data

In [ ]:
customers = pd.read_csv('../data/raw/customers.csv')
orders = pd.read_csv('../data/raw/orders.csv')
items = pd.read_csv('../data/raw/order_items.csv')
products = pd.read_csv('../data/raw/products.csv')
payments = pd.read_csv('../data/raw/payments.csv')
reviews = pd.read_csv('../data/raw/reviews.csv')
sellers = pd.read_csv('../data/raw/sellers.csv')

print(f'Raw data loaded:')
for name, df in [('customers',customers),('orders',orders),('items',items),('products',products),('payments',payments),('reviews',reviews),('sellers',sellers)]:
    print(f'  {name}: {len(df):,} rows, {len(df.columns)} cols')

## 2. Schema Validation

In [ ]:
# Expected schemas
schemas = {
    'customers': ['customer_id','customer_unique_id','customer_zip_code','customer_city','customer_state'],
    'orders': ['order_id','customer_id','order_status','order_purchase_timestamp'],
    'items': ['order_id','order_item_id','product_id','seller_id','price','freight_value'],
}

for name, expected in schemas.items():
    df = eval(name)
    missing = set(expected) - set(df.columns)
    print(f'{name}: {"PASS" if not missing else f"MISSING: {missing}"}')

## 3. Missing Values

In [ ]:
for name, df in [('customers',customers),('orders',orders),('items',items),('reviews',reviews)]:
    missing = df.isnull().sum()
    total = len(df)
    print(f'\n{name.upper()} ({total:,} rows):')
    for col in df.columns:
        if missing[col] > 0:
            pct = missing[col] / total * 100
            print(f'  {col}: {missing[col]:,} missing ({pct:.1f}%)')

## 4. Duplicate Removal

In [ ]:
for name, df, key in [('customers',customers,'customer_id'),('orders',orders,'order_id')]:
    dupes = df.duplicated(subset=[key]).sum()
    print(f'{name}: {dupes} duplicates on {key}')
    if dupes > 0:
        df = df.drop_duplicates(subset=[key])
        print(f'  -> Removed, {len(df):,} rows remaining')

## 5. Date Standardization

In [ ]:
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
             'order_delivered_customer_date','order_estimated_delivery_date']

for col in date_cols:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors='coerce')
        print(f'{col}: converted, range [{orders[col].min()} to {orders[col].max()}]')

## 6. Price Validation

In [ ]:
# Remove zero/negative prices
items['price'] = pd.to_numeric(items['price'], errors='coerce')
before = len(items)
items = items[items['price'] > 0]
print(f'Price validation: removed {before - len(items)} invalid rows')
print(f'Price range: ${items["price"].min():.2f} to ${items["price"].max():.2f}')
print(f'Mean price: ${items["price"].mean():.2f}')

## 7. Save Cleaned Data

In [ ]:
# Save to processed directory
customers.to_csv('../data/processed/customers.csv', index=False)
orders.to_csv('../data/processed/orders.csv', index=False)
items.to_csv('../data/processed/order_items.csv', index=False)
products.to_csv('../data/processed/products.csv', index=False)
payments.to_csv('../data/processed/payments.csv', index=False)
reviews.to_csv('../data/processed/reviews.csv', index=False)
sellers.to_csv('../data/processed/sellers.csv', index=False)

print('All cleaned datasets saved to data/processed/')

## Summary
Data cleaning pipeline completed:
- Schema validated for all tables
- Missing values identified and handled
- Duplicates removed
- Dates standardized to datetime format
- Prices validated (removed zero/negative)
- Clean data saved to processed directory